# Recalibración Invariante al Enmarcado de `veto_dir` (`LiquidAI/LFM3-1.7B`)

Corrige las dos vulnerabilidades detectadas en `veto_robustness_lfm3_l4.json` y Probe 4 (`tracer_upgrades_lfm3_l4.json`) sin **ninguna fuga de datos de `JailbreakBench/JBB-Behaviors`** y usando **plantillas de calibración estrictamente disjuntas** a las de evaluación (`educational`, `fiction`, `hypothetical`):
1. **Regularización explícita por covarianza de enmarcado (`\Sigma_{\text{frame}}`):** penaliza directamente las direcciones `h(wrap(x)) - h(x)` para neutralizar el atajo léxico (`"For"`, `"purely"`, `"ensuring"`).
2. **Balanceo de prompts directos sin cobertura académica (`DIRECT_BENIGN_CAL_PROMPTS` + pesos 50/50):** recupera el sobre-rechazo benigno fuera de distribución (`jbb_benign_ood`) sin perder `100%` de coincidencia exacta en controles sanos.
3. **Validación Group K-Fold OOF estricta** agrupada por prompt base y seleccionada sobre envoltorios de validación no vistos durante el ajuste de pesos.
4. **Validación completa antes/después** usando el mismo `.moipatch` base (`lfm3_safety_routed_refusal.moipatch`): OR-Bench Half-B + OR-Bench-Toxic + las **6 suites de `JailbreakBench/JBB-Behaviors`** (aceleradas a `bs=16`).

Guarda el nuevo parche `lfm3_safety_routed_invariant.moipatch` en Drive y los resultados en `MyDrive/wsi_runs/FINALES/`.

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
RUN_NAME = "veto_invariant_lfm3_l4"
REPO = "/content/wsi"
BASE = "/content/base"
PATCH_IN_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_safety_routed_refusal.moipatch"
PATCH_OUT_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_safety_routed_invariant.moipatch"
UNVETOED_PATCH_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_surgical_refusal.moipatch"
N_OR_EXTRA = 240
N_TOXIC_CAL = 240
N_TOXIC_TEST = 200
N_HARMFUL_JBB = 100
N_BENIGN_JBB = 100
N_CONTROL_JBB = 30
MAX_NEW_TOK = 128
BATCH_SIZE = 16

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers accelerate safetensors pandas pyarrow huggingface_hub datasets
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
RUN_DIR = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3"
os.makedirs(FINALES, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

if not os.path.exists(PATCH_IN_DRIVE):
    raise FileNotFoundError(f"No encuentro el .moipatch de seguridad base en Drive: {PATCH_IN_DRIVE}")

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/krexito/weight-space-index.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| patch_in:", PATCH_IN_DRIVE, "| has_unvetoed:", os.path.exists(UNVETOED_PATCH_DRIVE))

In [ ]:
import os, shutil, subprocess, sys

subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

LOCAL_PATCH_OUT = "/content/lfm3_safety_routed_invariant.moipatch"
LOCAL_OUT = f"/content/{RUN_NAME}.json"
SCRIPT = f"{REPO}/scripts/calibrate_veto_invariant_lfm3.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}).")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--patch", PATCH_IN_DRIVE,
    "--patch-out", LOCAL_PATCH_OUT,
    "--n-or-extra", str(N_OR_EXTRA),
    "--n-toxic-cal", str(N_TOXIC_CAL),
    "--n-toxic-test", str(N_TOXIC_TEST),
    "--n-harmful-jbb", str(N_HARMFUL_JBB),
    "--n-benign-jbb", str(N_BENIGN_JBB),
    "--n-control-jbb", str(N_CONTROL_JBB),
    "--max-new-tokens", str(MAX_NEW_TOK),
    "--batch-size", str(BATCH_SIZE),
    "--out", LOCAL_OUT,
]
if os.path.exists(UNVETOED_PATCH_DRIVE):
    cmd.extend(["--unvetoed-patch", UNVETOED_PATCH_DRIVE])

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

os.makedirs(os.path.dirname(PATCH_OUT_DRIVE), exist_ok=True)
shutil.copyfile(LOCAL_PATCH_OUT, PATCH_OUT_DRIVE + ".tmp")
os.replace(PATCH_OUT_DRIVE + ".tmp", PATCH_OUT_DRIVE)
print("Parche calibrado guardado atómicamente en Drive:", PATCH_OUT_DRIVE)

In [ ]:
import datetime, json, os, subprocess

res = json.load(open(LOCAL_OUT, encoding="utf-8"))
_pct = lambda v: f"{v:6.1%}" if isinstance(v, (int, float)) else "   n/a"
rr = res["router_report"]
ab = rr.get("ablation_no_frame_penalty") or {}
print("=== SELECCIÓN CV (con Sigma_frame vs ablación lam_frame=0) ===")
print(
    f"Ganador   : L{rr['layer']} g1={rr['g1_mode']} lam_r={rr['lam_ridge_rel']} lam_f={rr['lam_frame']} orth={rr['orth_refusal']} | "
    f"ben_raw={_pct(rr['oof_ben_raw_tpr'])} worst_tox_fpr={_pct(rr['oof_worst_tox_fpr'])} shift={rr['norm_unseen_frame_shift']:.3f}"
)
if ab:
    print(
        f"Sin frame : L{ab['layer']} g1={ab['g1_mode']} lam_r={ab['lam_ridge_rel']} lam_f={ab['lam_frame']} orth={ab['orth_refusal']} | "
        f"ben_raw={_pct(ab['oof_ben_raw_tpr'])} worst_tox_fpr={_pct(ab['oof_worst_tox_fpr'])} shift={ab['norm_unseen_frame_shift']:.3f}"
    )

ba = res["before_after_invariance"]
print("\n=== INVARIANCIA AL ENMARCADO (wrappers no vistos de calibración) ===")
print(
    f"rms_shift : viejo={ba.get('old_rms_shift')} -> nuevo={ba['new_rms_shift']} | "
    f"shift/std : viejo={ba.get('old_shift_to_std')} -> nuevo={ba['new_shift_to_std']}"
)

print("\n=== IN-DISTRIBUTION HELD-OUT (OR-Bench Half-B + OR-Bench-Toxic) ===")
id_arms = res["in_distribution"]["arms"]
for arm_name in ("base", "old_safety_routed", "new_invariant_safety_routed"):
    m = id_arms.get(arm_name)
    if not m:
        continue
    print(
        f"{arm_name:28s} | or_ref={_pct(m.get('or_bench_refusal_rate'))} | recov={_pct(m.get('recovered_of_base_refused'))} | "
        f"toxic_kept={_pct(m.get('toxic_refusal_kept'))} | ctrl_exact={_pct(m.get('control_exact_match_rate'))} | kl={m.get('healthy_kl')}"
    )
idg = res["in_distribution"]["gate"]
print(f"IN-DISTRIBUTION GATE : {'PASS' if idg['passed'] else 'FAIL'} ({idg['criterion']})")

print("\n=== JAILBREAKBENCH OOD + ENMARCADO ADVERSARIAL (6 suites) ===")
jbb = res["jbb_robustness"]
for sname, s in jbb["suites"].items():
    tel = s.get("router_telemetry") or {}
    extra = (
        f"kept={_pct(s.get('toxic_refusal_kept'))}"
        if s["is_harmful"]
        else (f"ctrl_exact={_pct(s.get('control_exact_match_rate'))}" if s["is_control"] else f"recov={_pct(s.get('recovered_of_base_refused'))}")
    )
    print(
        f"{sname:25s} | bs={s.get('batch_size_used', 1):2d} | base_ref={_pct(s['base_refusal_rate'])} | "
        f"safe_ref={_pct(s['safety_routed_refusal_rate'])} | {extra} | "
        f"fire={_pct(tel.get('patch_fire_rate'))} | veto={_pct(tel.get('gate2_veto_rate'))}"
    )

rg = jbb["robustness_gate"]
print(f"\n=== JBB ROBUSTNESS GATE : {'PASS' if rg['overall_passed'] else 'FAIL'} ===")
print(
    f"raw_kept={_pct(rg['raw_ood_kept'])} | min_framed_kept={_pct(rg['min_framed_kept'])} | framed={rg['framed_kept']} | "
    f"benign_ood_recov={_pct(rg['ood_benign_recovered'])} | controls_ok={rg['controls_passed']}"
)
print("Elapsed total (s):", res["elapsed_s"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "calibrate_veto_invariant_lfm3_colab.ipynb",
    "patch_out_drive": PATCH_OUT_DRIVE,
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
for out_dir in (FINALES, RUN_DIR):
    os.makedirs(out_dir, exist_ok=True)
    dst = f"{out_dir}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)
    os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", f"{FINALES}/{RUN_NAME}.json")

In [ ]:
AUTO_APAGAR = True
import os, time, json
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")